In [1]:
import os
import sys
import gc
import json
import time
import random
import re
import types
import signal
import contextlib
import subprocess
import textwrap
import importlib.util
import importlib.machinery
from typing import Any, Dict, List, Optional, Tuple

# ============================================================
# Kaggle-safe bootstrap
# ============================================================

def _stub_torchaudio() -> None:
    """Avoid torchaudio import crashes in some Kaggle images."""
    if "torchaudio" in sys.modules:
        return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    submods = ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]
    sys.modules["torchaudio"] = ta
    for sm in submods:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m

_stub_torchaudio()
sys.modules["torchvision"] = None

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])

for mod, pip in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),
    ("transformers", "transformers<=4.48.3"),
    ("accelerate", "accelerate"),
    ("sentencepiece", "sentencepiece"),
    ("peft", "peft"),
]:
    ensure_pkg(mod, pip)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from datasets import load_dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# Monkeypatch torch compiler to avoid HF v4.49 kwargs crash
if hasattr(torch, "compiler") and hasattr(torch.compiler, "disable"):
    _original_disable = torch.compiler.disable
    def _safe_disable(fn=None, recursive=True, **kwargs):
        kwargs.pop("reason", None) 
        return _original_disable(fn=fn, recursive=recursive)
    torch.compiler.disable = _safe_disable

try:
    from transformers.cache_utils import DynamicCache
    if not hasattr(DynamicCache, "from_legacy_cache"):
        @classmethod
        def _from_legacy_cache(cls, past_key_values=None):
            return cls()
        DynamicCache.from_legacy_cache = _from_legacy_cache
except Exception:
    pass

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
CHECKPOINTS_DIR = "/kaggle/input/datasets/adithyal2/phi3-adapterss/hierarchical_steering_grpo_phi3/checkpoints"
CHECKPOINT_NAMES = [
    "checkpoint-150",
    "checkpoint-200",
    "checkpoint-250",
    "final_model"
]

OUT_DIR = "/kaggle/working/phi3_residual_stream_deep_insights"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
HTML_DIR = os.path.join(OUT_DIR, "html")
REPORT_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
for p in [OUT_DIR, CSV_DIR, PLOT_DIR, HTML_DIR, REPORT_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE_COUNT = torch.cuda.device_count()
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# Examples per benchmark
N_GSM8K = 2
N_STRATEGYQA = 2
N_MMLU = 2
MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
    "formal_logic",
    "high_school_mathematics",
]

MAX_NEW_TOKENS = {
    "gsm8k": 256,
    "strategyqa": 192,
    "mmlu": 192,
}

VERBOSE = True

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 200,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif",
})

# ============================================================
# SYSTEM / FEW-SHOT PROMPTS (SFT Training Equivalents)
# ============================================================

SYSTEM_PROMPT = (
    "You are a careful reasoning assistant. "
    "Think step by step inside <think>...</think>, "
    "then state ONLY your final answer inside <answer>...</answer>."
)

MMLU_FEWSHOT = (
    "Question: What is the capital of France?\n"
    "A. London\nB. Paris\nC. Rome\nD. Berlin\n"
    "<think>The capital of France is Paris.</think>\n"
    "<answer>B</answer>\n\n"
    "Question: Which number is prime?\n"
    "A. 4\nB. 6\nC. 7\nD. 9\n"
    "<think>7 has no divisors other than 1 and itself.</think>\n"
    "<answer>C</answer>\n\n"
    "Question: What is 2 to the power of 5?\n"
    "A. 10\nB. 16\nC. 25\nD. 32\n"
    "<think>2^5 = 32.</think>\n"
    "<answer>D</answer>\n\n"
)

GSM8K_FEWSHOT = (
    "Question: Natalia sold clips to 48 friends in April, and half as many in May. "
    "How many clips did she sell altogether?\n"
    "<think>April = 48. May = 48 / 2 = 24. Total = 48 + 24 = 72.</think>\n"
    "<answer>72</answer>\n\n"
    "Question: Weng earns $12 an hour for babysitting. Yesterday she babysat for 50 minutes. "
    "How much did she earn?\n"
    "<think>Per minute = 12 / 60 = 0.2. For 50 minutes = 0.2 * 50 = 10.</think>\n"
    "<answer>10</answer>\n\n"
    "Question: Betty wants a $100 wallet and has half the money. Parents give $15 and "
    "grandparents give twice the parents. How much more does she need?\n"
    "<think>Has 100/2 = 50. Parents 15. Grandparents 30. Total 95. Needs 5.</think>\n"
    "<answer>5</answer>\n\n"
    "Question: James writes a 3-page letter to 2 friends twice a week. How many pages a year?\n"
    "<think>Each time 3*2 = 6. Twice a week 12. Year 12*52 = 624.</think>\n"
    "<answer>624</answer>\n\n"
)

STRATEGYQA_FEWSHOT = (
    "Answer each yes/no question with a single word: yes or no.\n\n"
    "Question: Would a vegetarian eat a hamburger made of beef?\n"
    "Answer: no\n\n"
    "Question: Is the Atlantic Ocean larger than the country of Australia?\n"
    "Answer: yes\n\n"
    "Question: Can a person survive without drinking any water for a month?\n"
    "Answer: no\n\n"
    "Question: Does the Sun rise in the east?\n"
    "Answer: yes\n\n"
)

def log(msg: str) -> None:
    if VERBOSE:
        print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def gpu_report(prefix: str = "GPU") -> None:
    if not torch.cuda.is_available():
        log("GPU report: CUDA unavailable.")
        return
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)
        log(f"{prefix} {i}: allocated={alloc:.2f} GB reserved={reserved:.2f} GB")

# ============================================================
# UTILS
# ============================================================

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

@contextlib.contextmanager
def timer(label: str):
    start = time.time()
    log(f"{label} ...")
    try:
        yield
    except Exception as e:
        log(f"{label} failed after {time.time() - start:.1f}s: {type(e).__name__}: {e}")
        raise
    else:
        log(f"{label} done in {time.time() - start:.1f}s")

@contextlib.contextmanager
def time_limit(seconds: int, label: str):
    if seconds <= 0 or not hasattr(signal, "SIGALRM"):
        yield
        return
    def _handler(signum, frame):
        raise TimeoutError(f"{label} timed out after {seconds}s")
    old_handler = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def json_default(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return str(x)

def save_df(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)
    log(f"Saved CSV -> {path}")

def save_json(obj: Any, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=json_default)
    log(f"Saved JSON -> {path}")

def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    log(f"Saved figure -> {path}")

def preview_text(s: str, max_len: int = 110) -> str:
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def text_wrap(text, width=80):
    if text is None: return "None"
    return "\n".join(textwrap.wrap(str(text), width=width))

def sample_dataset(ds, n: int, seed: int, tag: str):
    n = min(n, len(ds))
    cache_path = os.path.join(CACHE_DIR, f"{tag}_n{n}_seed{seed}.json")
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            idx = json.load(f)
        log(f"Loaded cached sample indices for {tag} ({len(idx)} items).")
        return ds.select(idx), idx
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(ds), size=n, replace=False).tolist()
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(idx, f)
    log(f"Sampled {len(idx)} items for {tag}.")
    return ds.select(idx), idx

# ============================================================
# STRING / ANSWER PARSERS
# ============================================================

def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None:
        return None
    try:
        x = float(str(pred).replace(",", ""))
        if abs(x - round(x)) < 1e-8:
            return str(int(round(x)))
        return str(x)
    except Exception:
        return None

def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try:
        return abs(float(pred) - float(gold)) <= 1e-6
    except Exception:
        return False

def extract_number(text: str) -> Optional[str]:
    if not text:
        return None
    # 1. Isolate the text inside <answer>...</answer>
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    
    # 2. Extract numbers from the isolated span
    nums = re.findall(r"-?\d+\.?\d*", span.replace(",", ""))
    if nums:
        return nums[-1]
        
    # 3. Fallback for GSM8K standard format
    hash_match = re.findall(r"####\s*(-?\d+\.?\d*)", text.replace(",", ""))
    if hash_match:
        return hash_match[-1]
    return None

def extract_yes_no(text: str) -> Optional[str]:
    if not text:
        return None
    # 1. Isolate the text inside <answer>...</answer>
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    
    # 2. Exact whole-word matches
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
        
    # 3. Ultimate fallback
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    return None

def extract_mcq(text: str) -> Optional[str]:
    if not text:
        return None
    # 1. Isolate the text inside <answer>...</answer>
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span_up = span.upper().strip()
    
    if span_up in ["A", "B", "C", "D"]:
        return span_up
        
    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"<ANSWER>\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()
            
    hits = re.findall(r"\b([ABCD])\b", span_up)
    return hits[-1].upper() if hits else None

def explicit_answer_present(text: str) -> bool:
    return bool(re.search(r"\b(yes|no|[ABCD])\b", str(text), flags=re.IGNORECASE))

def normalize_choice_order(choices: List[str], perm: Tuple[int, int, int, int]):
    return [choices[i] for i in perm]

def canonical_letter_from_perm_answer(pred_letter: Optional[str], perm: Tuple[int, int, int, int]) -> Optional[str]:
    if pred_letter is None:
        return None
    pred_letter = pred_letter.upper().strip()
    if pred_letter not in ["A", "B", "C", "D"]:
        return None
    idx = ord(pred_letter) - 65
    return chr(65 + perm[idx])

def get_mmlu_choices(sample: Dict[str, Any]) -> List[str]:
    choices = sample.get("choices")
    if choices is None:
        return []
    if isinstance(choices, str):
        try:
            parsed = json.loads(choices)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    if isinstance(choices, list):
        return choices
    try:
        return list(choices)
    except Exception:
        return []

def safe_mmlu_gold(sample: Dict[str, Any]) -> Optional[str]:
    g = sample.get("gold")
    if g is None:
        return None
    g = str(g).strip().upper()
    return g if g in ["A", "B", "C", "D"] else None

def token_ids_for_text(tokenizer, text: str) -> List[int]:
    forms = [text, f" {text}", f"\n{text}", text.upper(), f" {text.upper()}", text.lower(), f" {text.lower()}"]
    ids = []
    for form in forms:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1:
            ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) > 0:
            ids.append(enc[0])
    return sorted(set(ids))

def topn_token_strings(tokenizer, logits: torch.Tensor, n: int = 5) -> List[Tuple[str, float]]:
    probs = torch.softmax(logits.float(), dim=-1)
    vals, ids = torch.topk(probs, k=min(n, probs.shape[-1]))
    return [(tokenizer.decode([i]).strip(), float(p)) for p, i in zip(vals.tolist(), ids.tolist())]

def tensor_prob_for_ids(logits: torch.Tensor, ids: List[int]) -> float:
    if len(ids) == 0:
        return float("nan")
    probs = torch.softmax(logits.float(), dim=-1)
    return float(max([probs[i].item() for i in ids]))

def tensor_logit_for_ids(logits: torch.Tensor, ids: List[int]) -> float:
    if len(ids) == 0:
        return float("nan")
    return float(max([logits.float()[i].item() for i in ids]))

def candidate_tokens(tokenizer, sample: Dict[str, Any], final_logits: torch.Tensor, pred: Optional[str]) -> List[str]:
    toks = []
    if sample["task"] == "strategyqa":
        toks += ["yes", "no"]
    elif sample["task"] == "mmlu":
        toks += ["A", "B", "C", "D"]
    elif sample["task"] == "gsm8k":
        if sample.get("gold"):
            toks.append(str(sample["gold"]))
        if pred:
            toks.append(str(pred))
            
    # Always include structural tokens
    toks += ["<think>", "<answer>"]
    toks += [t for t, _ in topn_token_strings(tokenizer, final_logits, n=5)]
    
    uniq = []
    for t in toks:
        if t and t not in uniq and len(t.strip()) > 0:
            uniq.append(t)
    return uniq[:10]

def np_pca_2d(X: np.ndarray):
    X = np.asarray(X, dtype=np.float64)
    Xc = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    Z = Xc @ Vt[:2].T
    return Z, Vt[:2]

# ============================================================
# CHAT TEMPLATE PROMPT INJECTION
# ============================================================

def build_task_prompt(tokenizer, sample: Dict[str, Any]) -> str:
    """Uses proper Phi-3 Chat Template exactly mimicking the working SFT evaluation code."""
    task = sample["task"]
    question = sample["question"]
    
    if task == "gsm8k":
        user_content = (
            "Solve the maths problem step by step. "
            "Put ONLY the final numeric answer inside <answer></answer>.\n\n"
            f"{GSM8K_FEWSHOT}Question:\n{question}"
        )
    elif task == "strategyqa":
        user_content = f"{STRATEGYQA_FEWSHOT}Question: {question}\nAnswer:"
    elif task == "mmlu":
        choices = sample.get("choices", [])
        opts = "\n".join([f"{chr(65 + i)}. {c}" for i, c in enumerate(choices[:4])])
        user_content = (
            "Choose the single best option. "
            "Write exactly ONE letter (A, B, C or D) inside <answer></answer>.\n\n"
            f"{MMLU_FEWSHOT}Question:\n{question}\n\n{opts}"
        )
    else:
        raise ValueError(f"Unknown task: {task}")
        
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    
    # We let the tokenizer natively apply the formatting without manual <think> injections
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def model_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cpu")

def module_device(module) -> torch.device:
    try:
        return next(module.parameters()).device
    except Exception:
        return torch.device("cpu")

def module_dtype(module) -> torch.dtype:
    try:
        return next(module.parameters()).dtype
    except Exception:
        return torch.float32

# ============================================================
# DATA LOADING
# ============================================================

def load_gsm8k_dataset():
    for repo in ["gsm8k", "openai/gsm8k"]:
        try:
            log(f"Loading GSM8K from {repo} ...")
            ds = load_dataset(repo, "main", split="test")
            log(f"Loaded GSM8K ({repo}) with {len(ds)} rows.")
            return ds, repo
        except Exception as e:
            log(f"GSM8K load failed for {repo}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load GSM8K from the available dataset names.")

def load_strategyqa_dataset():
    log("Loading StrategyQA test split ...")
    ds = load_dataset("ChilleD/StrategyQA", split="test")
    log(f"Loaded StrategyQA with {len(ds)} rows.")
    return ds

def load_mmlu_subject(subject: str):
    for split in ["test", "validation", "dev"]:
        try:
            log(f"Loading MMLU subject={subject} split={split} ...")
            ds = load_dataset("cais/mmlu", subject, split=split)
            log(f"Loaded MMLU subject={subject} split={split} with {len(ds)} rows.")
            return ds, split
        except Exception as e:
            log(f"MMLU load failed for {subject}/{split}: {type(e).__name__}: {e}")
    raise RuntimeError(f"Could not load MMLU subject {subject}")

def build_samples(tokenizer) -> List[Dict[str, Any]]:
    samples = []

    gsm, gsm_repo = load_gsm8k_dataset()
    gsm, gsm_idx = sample_dataset(gsm, N_GSM8K, SEED, "gsm8k_test")
    for i, s in enumerate(gsm):
        gold = canonical_number(str(s["answer"]).split("####")[-1].strip())
        samples.append({
            "sample_id": f"gsm8k_{i}",
            "task": "gsm8k",
            "subject": "gsm8k",
            "question": s["question"],
            "gold": gold,
            "prompt": build_task_prompt(tokenizer, {"task": "gsm8k", "question": s["question"]}),
            "source_index": int(gsm_idx[i]),
            "source_repo": gsm_repo,
        })

    qa = load_strategyqa_dataset()
    qa, qa_idx = sample_dataset(qa, N_STRATEGYQA, SEED, "strategyqa_test")
    for i, s in enumerate(qa):
        gold = "yes" if bool(s["answer"]) else "no"
        samples.append({
            "sample_id": f"strategyqa_{i}",
            "task": "strategyqa",
            "subject": "strategyqa",
            "question": s["question"],
            "gold": gold,
            "prompt": build_task_prompt(tokenizer, {"task": "strategyqa", "question": s["question"]}),
            "source_index": int(qa_idx[i]),
        })

    mmlu_candidates = []
    for subject in MMLU_SUBJECTS:
        try:
            ds, split_used = load_mmlu_subject(subject)
        except Exception:
            continue
        ds, idx = sample_dataset(ds, 1, SEED, f"mmlu_{subject}_{split_used}")
        for i, s in enumerate(ds):
            choices = get_mmlu_choices(s)
            gold = safe_mmlu_gold(s)
            if len(choices) >= 4 and gold is not None:
                mmlu_candidates.append({
                    "sample_id": f"mmlu_{subject}_{i}",
                    "task": "mmlu",
                    "subject": subject,
                    "question": s["question"],
                    "choices": choices[:4],
                    "gold": gold,
                    "prompt": build_task_prompt(tokenizer, {"task": "mmlu", "question": s["question"], "choices": choices[:4]}),
                    "source_index": int(idx[i]),
                    "split_used": split_used,
                })
    if len(mmlu_candidates) > N_MMLU:
        random.shuffle(mmlu_candidates)
        mmlu_candidates = mmlu_candidates[:N_MMLU]
    samples.extend(mmlu_candidates)

    random.shuffle(samples)
    return samples

# ============================================================
# MODEL LOADING
# ============================================================

def build_max_memory_dict() -> Dict[str, str]:
    if DEVICE_COUNT == 0:
        return {"cpu": "64GiB"}
    out = {}
    for i in range(DEVICE_COUNT):
        total_gib = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        out[i] = f"{max(8, int(total_gib * 0.88))}GiB"
    out["cpu"] = "64GiB"
    return out

MAX_MEMORY = build_max_memory_dict()

def has_adapter(path: str) -> bool:
    if not os.path.isdir(path): return False
    files = set(os.listdir(path))
    return any(x in files for x in ["adapter_config.json", "adapter_model.safetensors", "adapter_model.bin"])

def load_tokenizer(model_name: str = BASE_MODEL):
    log(f"Loading tokenizer for {model_name} ...")
    # Loaded directly natively without trust_remote_code
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    log("Tokenizer loaded.")
    return tok

def load_hf_model(model_source: str):
    load_kwargs = dict(
        low_cpu_mem_usage=True,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE_COUNT > 0 else None,
        max_memory=MAX_MEMORY if DEVICE_COUNT > 0 else None,
        attn_implementation="eager",
    )
    load_kwargs = {k: v for k, v in load_kwargs.items() if v is not None}

    # Loaded exactly as in the user's working inference script without config overrides
    with timer(f"HF load: {model_source}"):
        model = AutoModelForCausalLM.from_pretrained(model_source, **load_kwargs)

    model.eval()
    try:
        model.config.use_cache = False
        if getattr(model, "generation_config", None) is not None:
            model.generation_config.use_cache = False
    except Exception: pass

    return model

def merge_adapter_if_needed(base_model, adapter_path: str):
    if has_adapter(adapter_path) and HAS_PEFT:
        try:
            with timer(f"Merging adapter: {adapter_path}"):
                merged = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
                merged.eval()
            return merged
        except Exception as e:
            log(f"Adapter merge failed; using base model only. {type(e).__name__}: {e}")
    return base_model

def load_variant_model(source: str, tag: str):
    log(f"[{tag}] Loading model from: {source}")
    if os.path.isdir(source) and os.path.exists(os.path.join(source, "config.json")) and not has_adapter(source):
        hf_model = load_hf_model(source)
    else:
        base = load_hf_model(BASE_MODEL)
        hf_model = merge_adapter_if_needed(base, source)
    hf_model.eval()
    log(f"[{tag}] Model ready.")
    return hf_model, "hf"

# ============================================================
# GENERATION / RESIDUAL STREAM EXTRACTION
# ============================================================

@torch.inference_mode()
def generate_completion(model, tokenizer, prompt: str, task: str):
    log(f"Generating for task={task} (max_new_tokens={MAX_NEW_TOKENS[task]})")
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(model_device(model)) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    # Reverted to greedy exactly as in working reference code (do_sample=False)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS[task],
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full_output = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion, inputs["input_ids"], out

def run_forward(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(model_device(model)) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, return_dict=True)
    
    # Extract states for the final token across all layers
    resid_stack = [h[0, -1].detach().float().cpu() for h in out.hidden_states]
    final_logits = out.logits[0, -1].detach().float().cpu()
    
    # Extract the sequence-level residual stream at the final layer
    seq_resid = out.hidden_states[-1][0].detach().float().cpu()
    prompt_token_ids = inputs["input_ids"][0].cpu().tolist()
    
    del out
    del inputs
    torch.cuda.empty_cache()
    
    return prompt_token_ids, resid_stack, final_logits, seq_resid

def project_resid_to_logits(model, resid_vec: torch.Tensor):
    core = getattr(model, "model", model)
    norm = getattr(core, "norm", getattr(core, "final_layer_norm", getattr(model, "norm", None)))
    head = getattr(model, "lm_head", getattr(model, "embed_out", None))
    
    x = resid_vec.detach().clone()
    if x.ndim == 1: x = x.unsqueeze(0).unsqueeze(0)
    elif x.ndim == 2: x = x.unsqueeze(0)
        
    with torch.no_grad():
        if norm is not None:
            x = x.to(device=module_device(norm), dtype=module_dtype(norm))
            x = norm(x)
        if head is not None:
            x = x.to(device=module_device(head), dtype=module_dtype(head))
            logits = head(x)
        else:
            raise RuntimeError("lm_head not found")
            
    logits = logits[0, -1] if logits.ndim == 3 else logits[0]
    return logits.detach().float().cpu()

# ============================================================
# PARSING / METRICS & ANALYSIS
# ============================================================

def parse_prediction(sample: Dict[str, Any], completion: str):
    """Parses the generated completion and grades it against the gold label."""
    if sample["task"] == "gsm8k":
        pred = canonical_number(extract_number(completion))
        return pred, bool(is_number_correct(pred, sample["gold"]))
    if sample["task"] == "strategyqa":
        pred = extract_yes_no(completion)
        return pred, bool(pred == sample["gold"])
    if sample["task"] == "mmlu":
        pred = extract_mcq(completion)
        return pred, bool(pred == sample["gold"])
    return None, False

def analyze_sample(model, tokenizer, sample: Dict[str, Any], model_tag: str):
    prompt = sample["prompt"]
    full_output, completion, _, _ = generate_completion(model, tokenizer, prompt, sample["task"])
    pred, correct = parse_prediction(sample, completion)

    prompt_token_ids, resid_stack, final_logits, seq_resid = run_forward(model, tokenizer, prompt)
    final_probs = torch.softmax(final_logits, dim=-1)
    final_entropy = float(-(final_probs * torch.log_softmax(final_logits, dim=-1)).sum().item())
    
    cand = candidate_tokens(tokenizer, sample, final_logits, pred)
    cand_ids = {c: token_ids_for_text(tokenizer, c) for c in cand}

    norms = [float(torch.norm(r).item()) for r in resid_stack]
    diff_norms = [0.0] + [float(torch.norm(resid_stack[i] - resid_stack[i-1]).item()) for i in range(1, len(resid_stack))]
    
    final_vec = resid_stack[-1]
    cosines = [float(torch.dot(r, final_vec) / (torch.norm(r) * torch.norm(final_vec) + 1e-12)) for r in resid_stack]

    layer_rows = []
    layer_logits_list = []
    for layer_idx, resid in enumerate(resid_stack):
        l_logits = project_resid_to_logits(model, resid)
        layer_logits_list.append(l_logits)
        
        l_probs = torch.softmax(l_logits, dim=-1)
        top_tok, top_prob = topn_token_strings(tokenizer, l_logits, n=1)[0]
        entropy = float(-(l_probs * torch.log_softmax(l_logits, dim=-1)).sum().item())
        kl_to_final = float(torch.nn.functional.kl_div(torch.log_softmax(l_logits, dim=-1), final_probs, reduction='sum').item())
        
        row = {
            "model_tag": model_tag,
            "sample_id": sample["sample_id"],
            "task": sample["task"],
            "subject": sample["subject"],
            "layer": layer_idx,
            "resid_norm": norms[layer_idx],
            "diff_norm": diff_norms[layer_idx],
            "cosine_to_final": cosines[layer_idx],
            "top_token": top_tok,
            "top_prob": top_prob,
            "entropy": entropy,
            "kl_to_final": kl_to_final
        }
        for c, ids in cand_ids.items():
            row[f"prob_{c}"] = tensor_prob_for_ids(l_logits, ids)
            row[f"logit_{c}"] = tensor_logit_for_ids(l_logits, ids)
        layer_rows.append(row)
        
    layer_df = pd.DataFrame(layer_rows)

    target_ids = {
        "gold": token_ids_for_text(tokenizer, str(sample["gold"])),
        "pred": token_ids_for_text(tokenizer, str(pred or sample["gold"])),
    }

    summary = {
        "model_tag": model_tag,
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "subject": sample["subject"],
        "question": sample["question"],
        "gold": sample["gold"],
        "prediction": pred,
        "correct": bool(correct),
        "prompt": prompt,
        "full_output": full_output,
        "completion": completion,
        "completion_len": len(tokenizer.encode(completion, add_special_tokens=False)),
        "prompt_len": len(prompt_token_ids),
        "final_top_token": topn_token_strings(tokenizer, final_logits, n=1)[0][0],
        "final_top_prob": topn_token_strings(tokenizer, final_logits, n=1)[0][1],
        "final_entropy": final_entropy,
        "hedged": bool(re.search(r"\b(maybe|probably|perhaps|uncertain|not sure|depends)\b", completion.lower())),
        "format_ok": bool(explicit_answer_present(completion)),
        "candidate_tokens": json.dumps(cand),
        "gold_prob_final": tensor_prob_for_ids(final_logits, target_ids["gold"]),
        "pred_prob_final": tensor_prob_for_ids(final_logits, target_ids["pred"]),
        "gold_logit_final": tensor_logit_for_ids(final_logits, target_ids["gold"]),
        "pred_logit_final": tensor_logit_for_ids(final_logits, target_ids["pred"]),
        "n_layers": len(resid_stack),
    }

    log(f"Sample {sample['sample_id']} done | correct={correct} | pred={pred} | gold={sample['gold']}")
    return summary, layer_df, resid_stack, cand, layer_logits_list, seq_resid, prompt_token_ids

def analyze_model(model, tokenizer, samples: List[Dict[str, Any]], model_tag: str):
    sample_rows, layer_rows, artifacts = [], [], {}
    log(f"[{model_tag}] Starting analysis for {len(samples)} samples.")
    gpu_report(f"[{model_tag}] pre-analysis GPU")

    for i, sample in enumerate(samples):
        log(f"[{model_tag}] {i+1}/{len(samples)} | {sample['task']} | {preview_text(sample['question'], 90)}")
        
        summary, layer_df, resid_stack, cand, layer_logits, seq_resid, prompt_token_ids = analyze_sample(model, tokenizer, sample, model_tag)
        
        sample_rows.append(summary)
        layer_rows.append(layer_df)
        artifacts[sample["sample_id"]] = {
            "summary": summary, "layer_df": layer_df, "resid_stack": resid_stack, "cand": cand, 
            "logits": layer_logits, "seq_resid": seq_resid, "prompt_token_ids": prompt_token_ids
        }
        
        task_dir = os.path.join(PLOT_DIR, model_tag)
        os.makedirs(task_dir, exist_ok=True)
        plot_residual_dashboard(summary, layer_df, resid_stack, cand, model_tag, os.path.join(task_dir, f"{sample['sample_id']}_dashboard.png"))
        save_df(layer_df, os.path.join(CSV_DIR, f"{model_tag}_{sample['sample_id']}_layerwise.csv"))
        gpu_report(f"[{model_tag}] after sample {i+1}")
        free_memory()

    sample_df = pd.DataFrame(sample_rows)
    layer_df = pd.concat(layer_rows, ignore_index=True) if layer_rows else pd.DataFrame()
    log(f"[{model_tag}] Analysis complete.")
    return sample_df, layer_df, artifacts

# ============================================================
# RICH VISUALISATIONS
# ============================================================

def plot_residual_dashboard(summary: Dict[str, Any], layer_df: pd.DataFrame, resid_stack: List[torch.Tensor], candidates: List[str], model_tag: str, out_path: str):
    """Generates a rich 3x2 deep-insight dashboard for a single model pass."""
    layer_df = layer_df.sort_values("layer")
    layers = layer_df["layer"].tolist()
    
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.2)

    # 1. Norms & Layer Updates
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(layers, layer_df["resid_norm"], marker="o", color="#2c3e50", linewidth=2.5, label="Residual $L_2$ Norm")
    ax1.set_ylabel("Residual Norm", color="#2c3e50", fontweight='bold')
    ax1b = ax1.twinx()
    ax1b.bar(layers, layer_df["diff_norm"], color="#e74c3c", alpha=0.5, label="Update Magnitude $||h_l - h_{l-1}||$")
    ax1b.set_ylabel("Update Magnitude", color="#e74c3c", fontweight='bold')
    ax1.set_title("Activation Growth & Layer Contribution", fontweight='bold')
    ax1.legend(loc="upper left"); ax1b.legend(loc="upper right")
    
    # 2. Entropy & Confidence
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(layers, layer_df["entropy"], marker="^", color="#8e44ad", linewidth=2.5, label="Entropy")
    ax2.set_ylabel("Entropy (Bits)", color="#8e44ad", fontweight='bold')
    ax2b = ax2.twinx()
    ax2b.plot(layers, layer_df["top_prob"], marker="v", color="#27ae60", linewidth=2.5, label="Top-1 Confidence")
    ax2b.set_ylabel("Top-1 Probability", color="#27ae60", fontweight='bold')
    ax2.set_title("Information Collapse & Confidence", fontweight='bold')
    ax2.legend(loc="upper left"); ax2b.legend(loc="upper right")

    # 3. Logit Lens Heatmap
    ax3 = fig.add_subplot(gs[1, 0])
    cand_cols = [f"prob_{c}" for c in candidates if f"prob_{c}" in layer_df.columns]
    heat = layer_df[cand_cols].to_numpy().T if cand_cols else np.zeros((1, len(layers)))
    im = ax3.imshow(heat, aspect="auto", cmap="magma", origin="lower")
    ax3.set_yticks(range(len(candidates)))
    ax3.set_yticklabels(candidates)
    ax3.set_xlabel("Layer")
    ax3.set_title("Logit Lens: Candidate Token Probabilities", fontweight='bold')
    plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

    # 4. Trajectory to Final
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(layers, layer_df["cosine_to_final"], marker="o", color="#3498db", linewidth=2.5, label="Cosine to Final")
    ax4.set_ylabel("Cosine Similarity", color="#3498db", fontweight='bold')
    ax4b = ax4.twinx()
    ax4b.plot(layers, layer_df["kl_to_final"], marker="x", color="#f39c12", linewidth=2.5, linestyle="--", label="KL to Final")
    ax4b.set_ylabel("KL Divergence", color="#f39c12", fontweight='bold')
    ax4.set_title("Convergence to Final Distribution", fontweight='bold')
    ax4.legend(loc="upper left"); ax4b.legend(loc="upper right")

    # 5. PCA Trajectory
    ax5 = fig.add_subplot(gs[2, 0])
    X = np.stack([r.numpy() for r in resid_stack], axis=0)
    Z, _ = np_pca_2d(X)
    scatter = ax5.scatter(Z[:, 0], Z[:, 1], c=layers, cmap="cool", s=60, edgecolors="black", zorder=3)
    ax5.plot(Z[:, 0], Z[:, 1], color="gray", alpha=0.5, linewidth=1, zorder=2)
    for i, (x, y) in enumerate(Z):
        if i % 4 == 0 or i == len(Z)-1:
            ax5.annotate(str(i), (x, y), xytext=(5, 5), textcoords="offset points", fontsize=9, fontweight='bold')
    plt.colorbar(scatter, ax=ax5, label="Layer Depth")
    ax5.set_title("Residual Stream PCA Trajectory", fontweight='bold')

    # 6. Text Panel
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.axis("off")
    prompt_snippet = preview_text(summary["prompt"].split("<|user|>")[-1], 150)
    gen_text = text_wrap(summary["completion"], 70)
    text_content = (
        f"TASK: {summary['task'].upper()} | CORRECT: {summary['correct']}\n"
        f"GOLD: {summary['gold']} | PRED: {summary['prediction']}\n"
        f"{'-'*60}\n"
        f"PROMPT SNIPPET:\n{prompt_snippet}\n"
        f"{'-'*60}\n"
        f"GENERATION:\n{gen_text}\n"
        f"{'-'*60}\n"
        f"Format OK: {summary['format_ok']} | Final Entropy: {summary['final_entropy']:.2f}"
    )
    ax6.text(0.0, 1.0, text_content, ha='left', va='top', fontfamily='monospace', fontsize=10, 
             bbox=dict(boxstyle="round,pad=1", facecolor="#f8f9fa", edgecolor="#bdc3c7"))

    plt.suptitle(f"Deep Insight Dashboard | {summary['sample_id']} | {model_tag.upper()}", fontsize=20, fontweight='bold', y=0.95)
    save_plot(out_path)

def plot_comparison_dashboard(b_art: Dict, s_art: Dict, out_path: str):
    """Generates a rich 3x2 comparative dashboard between Base and SFT."""
    b_df, s_df = b_art["layer_df"], s_art["layer_df"]
    layers = b_df["layer"].tolist()
    
    fig = plt.figure(figsize=(20, 18))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.2)

    # 1. Divergence & Norms
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(layers, b_df["resid_norm"], marker="o", color="#7f8c8d", label="Base Norm")
    ax1.plot(layers, s_df["resid_norm"], marker="s", color="#2980b9", label="SFT Norm")
    ax1.set_ylabel("L2 Norm", fontweight='bold')
    ax1b = ax1.twinx()
    l2_dists = [float(torch.norm(b - s).item()) for b, s in zip(b_art["resid_stack"], s_art["resid_stack"])]
    ax1b.bar(layers, l2_dists, color="#c0392b", alpha=0.3, label="L2 Distance (Base vs SFT)")
    ax1b.set_ylabel("Divergence Magnitude", color="#c0392b", fontweight='bold')
    ax1.set_title("Base vs SFT Divergence", fontweight='bold')
    ax1.legend(loc="upper left"); ax1b.legend(loc="upper right")

    # 2. Entropy
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(layers, b_df["entropy"], marker="o", color="#7f8c8d", label="Base Entropy")
    ax2.plot(layers, s_df["entropy"], marker="s", color="#8e44ad", label="SFT Entropy")
    ax2.set_ylabel("Entropy (Bits)", fontweight='bold')
    ax2.set_title("Uncertainty Evolution", fontweight='bold')
    ax2.legend()

    # 3. Target Probability
    ax3 = fig.add_subplot(gs[1, 0])
    gold_tok = str(b_art["summary"]["gold"])
    gold_col = f"prob_{gold_tok}"
    if gold_col in b_df.columns:
        ax3.plot(layers, b_df[gold_col], marker="o", linestyle="--", color="#7f8c8d", label=f"Base (Gold: {gold_tok})")
        ax3.plot(layers, s_df[gold_col], marker="s", color="#27ae60", label=f"SFT (Gold: {gold_tok})")
    ax3.set_ylabel("Probability", fontweight='bold')
    ax3.set_title("Target Token Probability", fontweight='bold')
    ax3.legend()

    # 4. Logit Shift Heatmap (SFT - Base)
    ax4 = fig.add_subplot(gs[1, 1])
    cands = list(set(b_art["cand"]) | set(s_art["cand"]))
    valid_cols = [f"prob_{c}" for c in cands if f"prob_{c}" in b_df.columns and f"prob_{c}" in s_df.columns]
    
    if valid_cols:
        b_mat = b_df[valid_cols].to_numpy()
        s_mat = s_df[valid_cols].to_numpy()
        delta_mat = (s_mat - b_mat).T # Shape: (vocab, layers)
        
        max_abs = max(abs(delta_mat.min()), abs(delta_mat.max()), 0.01)
        cmap = LinearSegmentedColormap.from_list("custom_diverging", ["#2980b9", "white", "#c0392b"])
        im = ax4.imshow(delta_mat, aspect="auto", cmap=cmap, vmin=-max_abs, vmax=max_abs, origin="lower")
        ax4.set_yticks(range(len(valid_cols)))
        ax4.set_yticklabels([c.replace("prob_", "") for c in valid_cols])
        ax4.set_title("SFT Logit Shift (Red = Boosted, Blue = Suppressed)", fontweight='bold')
        plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

    # 5. PCA Overlay
    ax5 = fig.add_subplot(gs[2, 0])
    b_res, s_res = b_art["resid_stack"], s_art["resid_stack"]
    X = np.stack([r.numpy() for r in b_res + s_res], axis=0)
    Z, _ = np_pca_2d(X)
    Zb, Zs = Z[:len(b_res)], Z[len(b_res):]
    
    ax5.plot(Zb[:, 0], Zb[:, 1], marker="o", color="#7f8c8d", alpha=0.6, label="Base")
    ax5.plot(Zs[:, 0], Zs[:, 1], marker="s", color="#2980b9", alpha=0.9, label="SFT")
    ax5.set_title("PCA Trajectory Comparison", fontweight='bold')
    ax5.legend()

    # 6. Text Panel
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.axis("off")
    bsum, ssum = b_art["summary"], s_art["summary"]
    text_content = (
        f"SAMPLE ID: {bsum['sample_id']} | TASK: {bsum['task'].upper()}\n"
        f"{'-'*60}\n"
        f"BASE: Pred={bsum['prediction']} | Correct={bsum['correct']} | Entropy={bsum['final_entropy']:.2f}\n"
        f"SFT:  Pred={ssum['prediction']} | Correct={ssum['correct']} | Entropy={ssum['final_entropy']:.2f}\n"
        f"{'-'*60}\n"
        f"BASE GENERATION:\n{text_wrap(bsum['completion'], 70)}\n\n"
        f"SFT GENERATION:\n{text_wrap(ssum['completion'], 70)}"
    )
    ax6.text(0.0, 1.0, text_content, ha='left', va='top', fontfamily='monospace', fontsize=9, 
             bbox=dict(boxstyle="round,pad=1", facecolor="#f8f9fa", edgecolor="#bdc3c7"))

    plt.suptitle(f"Base vs SFT Comparative Dashboard | {bsum['sample_id']}", fontsize=20, fontweight='bold', y=0.95)
    save_plot(out_path)

def plot_sequence_comparison_dashboard(base_item: Dict[str, Any], sft_item: Dict[str, Any], base_seq_resid: torch.Tensor, sft_seq_resid: torch.Tensor, prompt_token_ids: List[int], tokenizer, out_path: str):
    """Token-by-Token Sequence Dynamics Dashboard"""
    log(f"Rendering sequence dynamics dashboard for {base_item['sample_id']} ...")
    
    b_norms = torch.norm(base_seq_resid, dim=-1).numpy()
    s_norms = torch.norm(sft_seq_resid, dim=-1).numpy()
    divergence = torch.norm(base_seq_resid - sft_seq_resid, dim=-1).numpy()
    
    b_cos = [torch.cosine_similarity(base_seq_resid[i], base_seq_resid[i+1], dim=0).item() for i in range(len(base_seq_resid)-1)]
    s_cos = [torch.cosine_similarity(sft_seq_resid[i], sft_seq_resid[i+1], dim=0).item() for i in range(len(sft_seq_resid)-1)]
    
    seq_len = len(b_norms)
    x_positions = np.arange(seq_len)
    
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.2)
    
    # 1. Residual Norms
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(x_positions, b_norms, alpha=0.8, color="#7f8c8d", label="Base")
    ax1.plot(x_positions, s_norms, alpha=0.8, color="#2980b9", label="SFT")
    ax1.set_title("Sequence Dynamics: L2 Norm (Final Layer)", fontweight='bold')
    ax1.set_xlabel("Token Position")
    ax1.set_ylabel("L2 Norm")
    ax1.legend(frameon=True)
    
    # 2. Base vs SFT Divergence
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(x_positions, divergence, color="#c0392b", label="Divergence")
    ax2.set_title("Base vs SFT Divergence over Sequence", fontweight='bold')
    ax2.set_xlabel("Token Position")
    ax2.set_ylabel("L2 Distance")
    
    # Annotate top 5 divergence peaks
    if seq_len > 0:
        top_k = min(5, seq_len)
        peak_indices = np.argsort(divergence)[-top_k:]
        for idx in peak_indices:
            tok_str = repr(tokenizer.decode([prompt_token_ids[idx]]))
            ax2.annotate(tok_str, (idx, divergence[idx]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=8, rotation=45)
            
    # 3. Rate of Change
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(x_positions[:-1], b_cos, alpha=0.7, color="#7f8c8d", label="Base")
    ax3.plot(x_positions[:-1], s_cos, alpha=0.7, color="#2980b9", label="SFT")
    ax3.set_title("Token-to-Token Stability (Cosine Sim to Next)", fontweight='bold')
    ax3.set_xlabel("Token Position")
    ax3.set_ylabel("Cosine Similarity")
    ax3.legend(frameon=True)
    
    # 4. Top Divergence Tokens Bar Chart
    ax4 = fig.add_subplot(gs[1, 1])
    if seq_len > 0:
        top_k = min(10, seq_len)
        peak_indices = np.argsort(divergence)[-top_k:][::-1]
        tokens_str = [f"{i}: {repr(tokenizer.decode([prompt_token_ids[i]]))}" for i in peak_indices]
        div_vals = [divergence[i] for i in peak_indices]
        ax4.barh(tokens_str[::-1], div_vals[::-1], color="#e67e22")
        ax4.set_title("Top 10 Most Divergent Tokens (Base vs SFT)", fontweight='bold')
        ax4.set_xlabel("L2 Distance")
    
    # 5. Zoomed-in Divergence (Last 50 tokens)
    ax5 = fig.add_subplot(gs[2, 0])
    zoom_len = min(60, seq_len)
    zoom_x = x_positions[-zoom_len:]
    ax5.plot(zoom_x, divergence[-zoom_len:], color="#c0392b", marker=".")
    ax5.set_title("Divergence Zoom (Final Prompts Tokens)", fontweight='bold')
    ax5.set_xlabel("Token Position")
    ax5.set_ylabel("L2 Distance")
    
    for idx in zoom_x:
        tok_str = repr(tokenizer.decode([prompt_token_ids[idx]]).strip())
        if tok_str.strip(): 
            ax5.annotate(tok_str, (idx, divergence[idx]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=7, rotation=90)
            
    # 6. Prompt Context Snapshot
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.axis("off")
    context_str = tokenizer.decode(prompt_token_ids[-min(200, seq_len):])
    text_content = (
        f"PROMPT END CONTEXT (Last 200 Tokens):\n{'-'*60}\n"
        f"{text_wrap(context_str, 80)}"
    )
    ax6.text(0.0, 1.0, text_content, ha='left', va='top', fontfamily='monospace', fontsize=9, 
             bbox=dict(boxstyle="round,pad=1", facecolor="#f8f9fa", edgecolor="#bdc3c7"))
             
    plt.suptitle(f"Sequence Reading Dynamics | {base_item['sample_id']}", fontsize=20, fontweight='bold', y=0.95)
    save_plot(out_path)

def plot_global_summary_progression(all_dfs: Dict[str, pd.DataFrame]):
    """Plots accuracy progression across all loaded checkpoints."""
    records = []
    for tag, df in all_dfs.items():
        summary = df.groupby("task", as_index=False).agg(
            n=("correct", "count"),
            accuracy=("correct", "mean"),
            format_rate=("format_ok", "mean"),
            hedge_rate=("hedged", "mean"),
            avg_len=("completion_len", "mean"),
        )
        summary.insert(0, "model", tag)
        records.append(summary)

    full_df = pd.concat(records, ignore_index=True)
    save_df(full_df, os.path.join(CSV_DIR, "all_checkpoints_summary.csv"))

    plt.figure(figsize=(12, 7))
    models = list(all_dfs.keys())
    tasks = full_df["task"].unique()

    markers = ['o', 's', '^', 'D', 'v']
    colors = ['#2980b9', '#e74c3c', '#27ae60', '#8e44ad', '#f39c12']

    for i, task in enumerate(tasks):
        task_data = full_df[full_df["task"] == task].set_index("model").reindex(models)
        plt.plot(models, task_data["accuracy"], marker=markers[i%len(markers)], color=colors[i%len(colors)], linewidth=2.5, markersize=8, label=task.upper())
        
        for m in models:
            val = task_data.loc[m, "accuracy"]
            if pd.notna(val):
                plt.annotate(f"{val:.2f}", (m, val), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')

    plt.title("Task Accuracy Progression Over Training Checkpoints", fontweight='bold', fontsize=16)
    plt.ylabel("Accuracy", fontweight='bold')
    plt.xlabel("Model Checkpoint", fontweight='bold')
    plt.xticks(rotation=15)
    plt.ylim(-0.05, 1.05)
    plt.legend(frameon=True)
    save_plot(os.path.join(PLOT_DIR, "accuracy_progression.png"))

    return full_df

# ============================================================
# MAIN
# ============================================================

def main():
    print("=======================================================")
    print("PHI-3 RESIDUAL STREAM DASHBOARD (HF-SAFE VERSION)")
    print("=======================================================")
    log("Boot sequence started.")
    gpu_report("Boot GPU")

    with timer("Tokenizer load"):
        tokenizer = load_tokenizer(BASE_MODEL)

    with timer("Dataset building"):
        samples = build_samples(tokenizer)
    if len(samples) == 0:
        raise RuntimeError("No samples were loaded.")

    save_df(pd.DataFrame(samples), os.path.join(CSV_DIR, "selected_samples.csv"))
    log(f"Selected {len(samples)} samples: {[s['sample_id'] for s in samples]}")
    gpu_report("Post-dataset GPU")

    all_dfs = {}
    all_artifacts = {}
    all_layer_dfs = []

    # Base model.
    with timer("Base model load"):
        base_model, base_kind = load_variant_model(BASE_MODEL, "BASE")
    gpu_report("After base load")
    with timer("Base model analysis"):
        base_df, base_layer_df, base_artifacts = analyze_model(base_model, tokenizer, samples, "base")
    save_df(base_df, os.path.join(CSV_DIR, "base_sample_summary.csv"))
    save_df(base_layer_df, os.path.join(CSV_DIR, "base_layerwise.csv"))
    save_json({k: v["summary"] for k, v in base_artifacts.items()}, os.path.join(REPORT_DIR, "base_sample_summaries.json"))
    free_memory(base_model)
    
    all_dfs["base"] = base_df
    all_artifacts["base"] = base_artifacts
    all_layer_dfs.append(base_layer_df)

    # Process all SFT Checkpoints iteratively
    for ckpt_name in CHECKPOINT_NAMES:
        ckpt_path = os.path.join(CHECKPOINTS_DIR, ckpt_name)
        if not os.path.exists(ckpt_path):
            log(f"WARNING: Skipping {ckpt_name} - Path not found: {ckpt_path}")
            continue

        with timer(f"{ckpt_name} load"):
            ckpt_model, _ = load_variant_model(ckpt_path, ckpt_name)
        gpu_report(f"After {ckpt_name} load")
        with timer(f"{ckpt_name} analysis"):
            ckpt_df, ckpt_layer_df, ckpt_artifacts = analyze_model(ckpt_model, tokenizer, samples, ckpt_name)
        
        save_df(ckpt_df, os.path.join(CSV_DIR, f"{ckpt_name}_sample_summary.csv"))
        save_df(ckpt_layer_df, os.path.join(CSV_DIR, f"{ckpt_name}_layerwise.csv"))
        save_json({k: v["summary"] for k, v in ckpt_artifacts.items()}, os.path.join(REPORT_DIR, f"{ckpt_name}_sample_summaries.json"))
        
        all_dfs[ckpt_name] = ckpt_df
        all_artifacts[ckpt_name] = ckpt_artifacts
        all_layer_dfs.append(ckpt_layer_df)
        free_memory(ckpt_model)

    with timer("Global summary plotting"):
        full_summary_df = plot_global_summary_progression(all_dfs)

    compare_dir = os.path.join(PLOT_DIR, "comparisons")
    os.makedirs(compare_dir, exist_ok=True)
    
    for sample in samples:
        sid = sample["sample_id"]
        for ckpt_name in CHECKPOINT_NAMES:
            if ckpt_name not in all_artifacts:
                continue
            if sid not in all_artifacts["base"] or sid not in all_artifacts[ckpt_name]:
                continue
            
            log(f"Rendering comparison dashboard for {sid} (Base vs {ckpt_name}) ...")
            plot_comparison_dashboard(
                all_artifacts["base"][sid],
                all_artifacts[ckpt_name][sid],
                os.path.join(compare_dir, f"{sid}_comparison_{ckpt_name}.png"),
            )
            
            plot_sequence_comparison_dashboard(
                all_artifacts["base"][sid]["summary"],
                all_artifacts[ckpt_name][sid]["summary"],
                all_artifacts["base"][sid]["seq_resid"],
                all_artifacts[ckpt_name][sid]["seq_resid"],
                all_artifacts["base"][sid]["prompt_token_ids"],
                tokenizer,
                os.path.join(compare_dir, f"{sid}_sequence_dynamics_{ckpt_name}.png"),
            )

    merged_layers = pd.concat(all_layer_dfs, ignore_index=True)
    save_df(merged_layers, os.path.join(CSV_DIR, "all_models_layerwise.csv"))

    report_lines = [
        "# Phi-3 Residual Stream Multi-Checkpoint Report\n",
        f"- Base model: `{BASE_MODEL}`",
        f"- Checkpoints directory: `{CHECKPOINTS_DIR}`",
        f"- Models analyzed: `base`, " + ", ".join([f"`{c}`" for c in CHECKPOINT_NAMES if c in all_artifacts]),
        f"- Samples per benchmark: GSM8K={N_GSM8K}, StrategyQA={N_STRATEGYQA}, MMLU={N_MMLU}",
        "\n## Global Progression Summary\n",
        full_summary_df.to_markdown(index=False),
        "\n## Notes\n",
        "- Residual stream representations are visualised for multiple benchmarks.",
        "- Sequence dynamics plot token-by-token processing states in the final layer.",
        "- The code loads only one model at a time to avoid Kaggle OOMs, running through all checkpoints sequentially.",
        "- Comparison dashboards map 'base' vs each fine-tuned checkpoint.",
        "- Verbose timestamps and GPU memory reports are included around expensive steps.",
    ]
    with open(os.path.join(REPORT_DIR, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    log(f"Saved report -> {os.path.join(REPORT_DIR, 'report.md')}")

    save_json(
        {
            "base_model": BASE_MODEL,
            "checkpoints_dir": CHECKPOINTS_DIR,
            "samples": [s["sample_id"] for s in samples],
            "progression_summary": full_summary_df.to_dict(orient="records"),
        },
        os.path.join(REPORT_DIR, "summary.json"),
    )

    print("\n===== FULL PROGRESSION SUMMARY =====")
    print(full_summary_df.to_string(index=False))
    print("\nSaved outputs to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOT_DIR}")
    print(f"- HTML: {HTML_DIR}")
    print(f"- Reports: {REPORT_DIR}")

if __name__ == "__main__":
    main()

PHI-3 RESIDUAL STREAM DASHBOARD (HF-SAFE VERSION)
[20:10:47] Boot sequence started.
[20:10:47] Boot GPU 0: allocated=0.00 GB reserved=0.00 GB
[20:10:47] Boot GPU 1: allocated=0.00 GB reserved=0.00 GB
[20:10:47] Tokenizer load ...
[20:10:47] Loading tokenizer for microsoft/phi-3-mini-4k-instruct ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

[20:19:22] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_comparison_checkpoint-200.png
[20:19:22] Rendering sequence dynamics dashboard for gsm8k_0 ...
[20:19:24] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_sequence_dynamics_checkpoint-200.png
[20:19:24] Rendering comparison dashboard for gsm8k_0 (Base vs checkpoint-250) ...


/tmp/ipykernel_23/2121817199.py:299: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[20:19:26] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_comparison_checkpoint-250.png
[20:19:26] Rendering sequence dynamics dashboard for gsm8k_0 ...
[20:19:28] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_sequence_dynamics_checkpoint-250.png
[20:19:28] Rendering comparison dashboard for gsm8k_0 (Base vs final_model) ...


/tmp/ipykernel_23/2121817199.py:299: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[20:19:30] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_comparison_final_model.png
[20:19:30] Rendering sequence dynamics dashboard for gsm8k_0 ...
[20:19:32] Saved figure -> /kaggle/working/phi3_residual_stream_deep_insights/plots/comparisons/gsm8k_0_sequence_dynamics_final_model.png
[20:19:32] Saved CSV -> /kaggle/working/phi3_residual_stream_deep_insights/csv/all_models_layerwise.csv
[20:19:32] Saved report -> /kaggle/working/phi3_residual_stream_deep_insights/reports/report.md
[20:19:32] Saved JSON -> /kaggle/working/phi3_residual_stream_deep_insights/reports/summary.json

===== FULL PROGRESSION SUMMARY =====
         model       task  n  accuracy  format_rate  hedge_rate  avg_len
          base      gsm8k  2       0.5          0.5         0.0    103.0
          base strategyqa  2       1.0          1.0         0.0      1.0
checkpoint-150      gsm8k  2       0.5          0.5         0.0    103.0
checkpoint-150 strategyqa  2       1.0 